# **Week1_Day3**

### **Connection to the database**

In [1]:
# import the necessary labraries

import pandas as pd
import psycopg2
import warnings
warnings.filterwarnings("ignore")

In [2]:
# create the connection to the database via psycopg2, using the .connect method
conn = psycopg2.connect(
    dbname="neondb",
    user="neondb_owner",
    password="a9Am7Yy5r9_T7h4OF2GN",
    host="ep-falling-glitter-a5m0j5gk-pooler.us-east-2.aws.neon.tech",
    port="5432",
    sslmode="require"  ##ssl encryption on
)

# create the cursor from the connection
cur = conn.cursor()

In [3]:
# query to get all high school data
query = "SELECT * \
         FROM nyc_schools.high_school_directory; \
"


# execute the query and fetch the data into a pandas dataframe
df = pd.read_sql(query, conn) 

# show all columns
pd.set_option("display.max_columns", None)

# show the first 3 rows of the dataframe
df.head(3)

,dbn,school_name,borough,building_code,phone_number,fax_number,grade_span_min,grade_span_max,expgrade_span_min,expgrade_span_max,start_time,end_time,priority01,priority02,priority03,priority04,priority05,priority06,priority07,priority08,priority09,priority10,location,phone_number2,school_email,website,subway,bus,grades2018,finalgrades,total_students,extracurricular_activities,school_sports,attendance_rate,pct_stu_enough_variety,pct_stu_safe,school_accessibility_description,directions1,requirement1,requirement2,requirement3,requirement4,requirement5,program1,code1,interest1,method1,seats9ge1,grade9gefilledflag1,grade9geapplicants1,seats9swd1,grade9swdfilledflag1,grade9swdapplicants1,campus_name,building_borough,building_location,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta,zip_codes,community_districts,borough_boundaries,city_council_districts,police_precincts,primary_address_line_1,city,state_code,postcode,school_type,overview_paragraph,program_highlights,language_classes,advancedplacement_courses,online_ap_courses,online_language_courses,psal_sports_boys,psal_sports_girls,psal_sports_coed,partner_cbo,partner_hospital,partner_highered,partner_cultural,partner_nonprofit,partner_corporate,partner_financial,partner_other,addtl_info1,addtl_info2,se_services,ell_programs,number_programs,Location 1,Community Board,Council District,Census Tract,Zip Codes,Community Districts,Borough Boundaries,City Council Districts,Police Precincts
0,27Q260,Frederick Douglass Academy VI High School,Queens,Q465,718-471-2154,718-471-2890,9.0,12,NaN,NaN,7:45 AM,2:05 PM,Priority to Queens students or residents who a...,Then to New York City residents who attend an ...,Then to Queens students or residents,Then to New York City residents,,,,,,,None,None,None,http://schools.nyc.gov/schoolportals/27/Q260,A to Beach 25th St-Wavecrest,"Q113, Q22",None,None,412.0,"After-school Program, Book, Writing, Homework ...","Step Team, Modern Dance, Hip Hop Dance",None,None,None,Not Functionally Accessible,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Far Rockaway Educational Campus,None,None,None,None,None,None,None,4300730.0,4.157360e+09,Far Rockaway-Bayswater ...,None,None,None,None,None,8-21 Bay 25 Street,Far Rockaway,NY,11691,,Frederick Douglass Academy (FDA) VI High Schoo...,"Advisory, Graphic Arts Design, Teaching Intern...",Spanish,"Calculus AB, English Language and Composition,...","Biology, Physics B","French, Spanish","Basketball, Cross Country, Indoor Track, Outdo...","Basketball, Cross Country, Indoor Track, Outdo...",,,"Jamaica Hospital Medical Center, Peninsula Hos...","York College, Brooklyn College, St. John's Col...",,"Queens District Attorney, Sports and Arts Foun...","Replications, Inc.",Citibank,New York Road Runners Foundation (NYRRF),"Uniform Required: plain white collared shirt, ...","Extended Day Program, Student Summer Orientati...",This school will provide students with disabil...,ESL,1,"{'latitude': '40.601989336', 'longitude': '-73...",14,31,100802,20529,51,3,47,59
1,21K559,Life Academy High School for Film and Music,Brooklyn,K400,718-333-7750,718-333-7775,9.0,12,NaN,NaN,8:15 AM,3:00 PM,Priority to New York City residents who attend...,Then to New York City residents,,,,,,,,,None,None,None,http://schools.nyc.gov/schoolportals/21/K559,D to 25th Ave ; N to Ave U ; N to Gravesend - ...,"B1, B3, B4, B6, B64, B82",None,None,260.0,"Film, Music, Talent Show, Holiday Concert, Stu...",,None,None,None,Functionally Accessible,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Lafayette Educational Campus,None,None,None,None,None,None,None,3186454.0,3.068830e+09,Gravesend ...,None,None,None,None,None,2630 Benson Avenue,Brooklyn,NY,11214,,At Life Academy High School for Film and Music...,"College Now, iLEARN courses, Art and Film Prod...",Spanish,,"Biology, English Literature and Composition, E...",,"Basketball, Bowling, Indoor Track, Soccer, Sof...","Basketball

In [4]:
# show the dataframe info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435 entries, 0 to 434
Columns: 105 entries, dbn to Police Precincts
dtypes: float64(6), int64(1), object(98)
memory usage: 357.0+ KB


### **🧮 School Distribution**

**How many schools are there in each borough?**

In [5]:
# query to get the number of schools per borough
query_1 = "SELECT \
                hsd.borough, \
                COUNT(DISTINCT hsd.dbn) AS nber_schools \
           FROM nyc_schools.high_school_directory hsd \
           GROUP BY 1; \
"
# execute the query and fetch the data into nber_schools_x_borough dataframe
nber_schools_x_borough = pd.read_sql(query_1, conn)

# display the nber_schools_x_borough
nber_schools_x_borough


,borough,nber_schools
0,Bronx,118
1,Brooklyn,121
2,Manhattan,106
3,Queens,80
4,Staten Island,10


### **🎓 Language Learners**

**What is the average % of English Language Learners (ELL) per borough?**

In [6]:
# query to get the average English Language Learner (ELL) in percentage per borough
query_2_1 = """SELECT \
                hsd.borough, \
                AVG(sd.ell_percent) as avg_ell_percent \
             FROM nyc_schools.high_school_directory hsd \
             LEFT JOIN nyc_schools.school_demographics sd ON sd.dbn = hsd.dbn \
             GROUP BY 1;
"""
# execute the query and fetch the data into avg_ell_x_borough_2 dataframe
avg_ell_x_borough_2_1 = pd.read_sql(query_2_1, conn)

# display the avg_ell_x_borough_2
avg_ell_x_borough_2_1

,borough,avg_ell_percent
0,Brooklyn,NaN
1,Queens,NaN
2,Staten Island,NaN
3,Manhattan,7.5725
4,Bronx,NaN


### **🔗School supporting special needs**

**Using the data from the school demographics and high school directory, write a query to find the top 3 schools in each borough with the highest percentage of special education students (sped_percent)**

In [7]:
# query to get the special education percentage per borough
sped_percent = " \
           SELECT \
             hsd.borough, \
             sd.schoolyear, \
             sd.\"Name\", \
             sd.sped_percent \
           FROM nyc_schools.high_school_directory hsd \
           LEFT JOIN nyc_schools.school_demographics sd ON sd.dbn = hsd.dbn \
           ORDER BY 1, 2 ASC;\
"

# execute the query and fetch the data into sped_percent dataframe
df_sped_percent = pd.read_sql(sped_percent, conn)

# display the first few rows of df_sped_percent
df_sped_percent.head()

,borough,schoolyear,Name,sped_percent
0,Bronx,NaN,None,NaN
1,Bronx,NaN,None,NaN
2,Bronx,NaN,None,NaN
3,Bronx,NaN,None,NaN
4,Bronx,NaN,None,NaN


In [8]:
for borough in df_sped_percent["borough"].unique():
    top_school = df_sped_percent[df_sped_percent["borough"] == borough][["Name", "sped_percent"]].nlargest(3, "sped_percent")
    print(f"Borough :{borough},{top_school}\n\n")

Borough :Bronx,   Name  sped_percent
0  None           NaN
1  None           NaN
2  None           NaN


Borough :Brooklyn,     Name  sped_percent
118  None           NaN
119  None           NaN
120  None           NaN


Borough :Manhattan,                                Name  sped_percent
267  EAST SIDE COMMUNITY HIGH SCHOOL          28.8
272  EAST SIDE COMMUNITY HIGH SCHOOL          27.7
261  EAST SIDE COMMUNITY HIGH SCHOOL          26.7


Borough :Queens,     Name  sped_percent
378  None           NaN
379  None           NaN
380  None           NaN


Borough :Staten Island,     Name  sped_percent
458  None           NaN
459  None           NaN
460  None           NaN




In [9]:
# Query to get the top 3 special education students by borough
query_top_3_sped_x_borough = """\
WITH school_ranking_1 AS ( \
    SELECT \
         hsd.borough, \
         sd.schoolyear, \
         sd.\"Name\", \
         sd.sped_percent, \
         ROW_NUMBER() OVER (PARTITION BY hsd.borough, sd.schoolyear ORDER BY sd.sped_percent DESC NULLs LAST) AS sped_student_rank \
         FROM nyc_schools.high_school_directory hsd \
         LEFT JOIN nyc_schools.school_demographics sd ON sd.dbn = hsd.dbn \
), \

school_ranking_2 as ( \
    SELECT \
         borough, \
         schoolyear, \
         "Name", \
         sped_percent, \
         row_number() over (partition by borough order by sped_percent desc NULLs LAST) AS sped_student_rank_1 \
    FROM school_ranking_1 \
    WHERE sped_student_rank <= 3 \
) \
SELECT * FROM school_ranking_2 WHERE sped_student_rank_1 <= 3;
"""

# execute query_top_3_sped_x_borough and fetch the data into top_3_sped_x_borough dataframe
top_3_sped_x_borough = pd.read_sql(query_top_3_sped_x_borough, conn)

# Display the top 3 special education students by borough
top_3_sped_x_borough


,borough,schoolyear,Name,sped_percent,sped_student_rank_1
0,Bronx,NaN,None,NaN,1
1,Bronx,NaN,None,NaN,2
2,Bronx,NaN,None,NaN,3
3,Brooklyn,NaN,None,NaN,1
4,Brooklyn,NaN,None,NaN,2
5,Brooklyn,NaN,None,NaN,3
6,Manhattan,20092010.0,EAST SIDE COMMUNITY HIGH SCHOOL,28.8,1
7,Manhattan,20102011.0,EAST SIDE COMMUNITY HIGH SCHOOL,27.7,2
8,Manhattan,20082009.0,EAST SIDE COMMUNITY HIGH SCHOOL,26.7,3
9,Queens,NaN,None,NaN,1



## **Summary**

### **1. Overview**

**Objective:** 
Quick Exploration and Analysis of New York education data.


**Key Findings** 

| **Borough** | **Number of School**|
|---|---|
|Brooklyn|121|
|Bronx|118|
|Manhattan|106|
|Queens|80|
|Staten Island|10|


| **Borough** | **Average English Language Leaarners**|
|---|---|
|Brooklyn|Not Available|
|Bronx|Not Available
|Manhattan|2005-2006 --> **5.68**; 06-07 --> **5.66**; 07-08 --> **6.58**; 08-09 --> **5.88**; 09-10 --> **8.16**; 10-11 --> **10.74**; 11-12 --> **11.96**|
|Queens|Not Available|
|Staten Island|Not Available|

| **Borough** | **Top 3 Schools (Percentage of Special Education Students)**|
|---|---|
|Brooklyn|Not Available|
|Bronx|Not Available
|Manhattan|EAST SIDE COMMUNITY HIGH SCHOOL(28.8); EAST SIDE COMMUNITY HIGH SCHOOL(27.7); EAST SIDE COMMUNITY HIGH SCHOOL(26.7)|
|Queens|Not Available|
|Staten Island|Not Available|

### **2. Methodology and Data Source**

We used Python and SQL together to retrieve and analyse new York School data.
- **Data Origin:** SQL database "nyc_schools" via DBeaver
- **Tools Used:** Pandas, Psycopg2 and Warnings


### **Anomalies**

**The table "school_demographics" has data only about the borough of Manhattan**

### **Conclusion**

- **Limitation:** Due to the lack of data about 4 borough in the "school_demographics" table, we couldn't have more insightfull findings.
- **Recommendation:** Either collect school_demographics data about the 4 absent borough or do a more granular analysis on Manhattan school_demographics data.
